# 02 — Exploratory Data Analysis (All International Matches)

This notebook explores the full, cleaned match-level and team-level datasets
(all 49,215 played international matches, 1872-2026) before narrowing to the
FIFA World Cup in notebook 03.

We answer:
- How has the volume and scoring rate of international football changed over time?
- Which teams have been the most successful overall, and by which metric?
- What does home-field / neutral-venue advantage look like statistically?


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)

from data_cleaning import run_cleaning_pipeline
from feature_engineering import add_match_features, build_team_level, validate_team_level
from metrics import team_summary

cleaned = run_cleaning_pipeline()
matches = add_match_features(cleaned["results"])
teams = build_team_level(matches)
validate_team_level(matches, teams)
print(f"Matches: {len(matches):,} | Team-level rows: {len(teams):,}")


Matches: 49,215 | Team-level rows: 98,430


## Matches and scoring trends over time

In [2]:
yearly_matches = matches.groupby("year").size()
print("Matches in the busiest year:", yearly_matches.idxmax(), "->", yearly_matches.max())
print("Matches in 2025:", yearly_matches.get(2025, 'n/a'))
yearly_matches.tail(10)


Matches in the busiest year: 2024 -> 1229
Matches in 2025: 997


year
2017     924
2018     929
2019    1147
2020     347
2021    1115
2022     969
2023    1054
2024    1229
2025     997
2026     165
dtype: int64

In [3]:
avg_goals_by_decade = matches.groupby("decade")["total_goals"].mean()
draw_rate_by_decade = matches.groupby("decade")["is_draw"].mean()
pd.DataFrame({"avg_goals_per_match": avg_goals_by_decade, "draw_rate": draw_rate_by_decade})


,avg_goals_per_match,draw_rate
decade,,
1870,4.538462,0.153846
1880,5.581818,0.090909
1890,5.152542,0.152542
1900,4.182482,0.175182
1910,4.218182,0.160606
1920,3.884058,0.179952
1930,4.318814,0.159407
1940,4.344538,0.153661
1950,4.004240,0.183525


Average goals per match has drifted down from the high-scoring early decades
(friendlies between a handful of strong European/South American sides) toward a
more stable ~2.5-2.7 goals/match as the football world has globalized and more
evenly-matched fixtures have entered the dataset. Draw rate has been comparatively
stable across decades, generally in the 22-26% range.

## Most active teams, most wins, highest win rates

In [4]:
summary = team_summary(teams, min_matches=1)
summary.to_csv(PROJECT_ROOT / "outputs" / "tables" / "team_summary_all_matches.csv", index=False)

print("Most active teams by matches played:")
print(summary.sort_values("matches_played", ascending=False).head(10)[["team", "matches_played"]])


Most active teams by matches played:
           team  matches_played
54       Sweden            1099
17      England            1088
24    Argentina            1064
10       Brazil            1057
18      Germany            1029
28  South Korea            1005
68      Hungary            1004
40       Mexico            1000
78      Uruguay             968
47       France             933


In [5]:
print("Most wins (all-time):")
print(summary.sort_values("wins", ascending=False).head(10)[["team", "matches_played", "wins"]])


Most wins (all-time):
           team  matches_played  wins
10       Brazil            1057   670
17      England            1088   623
18      Germany            1029   597
24    Argentina            1064   588
54       Sweden            1099   541
28  South Korea            1005   536
40       Mexico            1000   511
47       France             933   476
26        Italy             891   475
68      Hungary            1004   470


In [6]:
MIN_MATCHES = 300
main_pool = summary[summary["matches_played"] >= MIN_MATCHES]
print(f"Teams with >= {MIN_MATCHES} career matches: {len(main_pool)} of {len(summary)}")
print()
print("Highest win rate (min. 300 matches):")
print(main_pool.sort_values("win_rate", ascending=False).head(10)[["team", "matches_played", "win_rate"]])


Teams with >= 300 career matches: 133 of 333

Highest win rate (min. 300 matches):
              team  matches_played  win_rate
10          Brazil            1057  0.633869
15           Spain             781  0.588988
18         Germany            1029  0.580175
17         England            1088  0.572610
20            Iran             610  0.568852
24       Argentina            1064  0.552632
28     South Korea            1005  0.533333
26           Italy             891  0.533109
36  Czech Republic             358  0.527933
29         Croatia             394  0.527919


**Important methodological note:** without the 300-match threshold, the win-rate
leaderboard is dominated by entities like Jersey, Guernsey, Padania, and the Basque
Country -- regional/sub-national football associations that are not full FIFA members
and play almost exclusively in minor regional tournaments (e.g. the Island Games)
against similarly small opposition. Their high win rates reflect a weak schedule, not
elite performance. The 300-match threshold is the point in this dataset where these
entities largely drop out, leaving recognizable, heavily-fixtured national teams. This
trade-off (and the sensitivity of the ranking to the threshold) is examined
quantitatively in notebook 04.

## Goal differential, dominant wins, attack/defense

In [7]:
print("Best average goal differential (min. 300 matches):")
print(main_pool.sort_values("avg_goal_diff", ascending=False).head(10)[["team", "avg_goal_diff", "goals_scored_per_match", "goals_conceded_per_match"]])


Best average goal differential (min. 300 matches):
           team  avg_goal_diff  goals_scored_per_match  \
10       Brazil       1.270577                2.173132   
17      England       1.227941                2.184743   
15        Spain       1.148528                2.046095   
20         Iran       1.095082                1.885246   
18      Germany       1.085520                2.248785   
45    Australia       0.948187                2.013817   
24    Argentina       0.886278                1.894737   
38  Netherlands       0.872292                2.096921   
28  South Korea       0.867662                1.777114   
26        Italy       0.771044                1.754209   

    goals_conceded_per_match  
10                  0.902554  
17                  0.956801  
15                  0.897567  
20                  0.790164  
18                  1.163265  
45                  1.065630  
24                  1.008459  
38                  1.224629  
28                  0.909453  


In [8]:
print("Most dominant wins (margin >= 3 goals, min. 300 matches):")
print(main_pool.sort_values("dominant_wins", ascending=False).head(10)[["team", "dominant_wins", "dominant_win_rate"]])
print()
print("Strongest attack (goals scored per match, min. 300 matches):")
print(main_pool.sort_values("goals_scored_per_match", ascending=False).head(5)[["team", "goals_scored_per_match"]])
print()
print("Strongest defense (goals conceded per match, min. 300 matches):")
print(main_pool.sort_values("goals_conceded_per_match", ascending=True).head(5)[["team", "goals_conceded_per_match"]])


Most dominant wins (margin >= 3 goals, min. 300 matches):
           team  dominant_wins  dominant_win_rate
17      England            252           0.231618
10       Brazil            251           0.237465
18      Germany            216           0.209913
54       Sweden            187           0.170155
24    Argentina            183           0.171992
38  Netherlands            172           0.196123
28  South Korea            168           0.167164
68      Hungary            168           0.167331
15        Spain            165           0.211268
40       Mexico            154           0.154000

Strongest attack (goals scored per match, min. 300 matches):
           team  goals_scored_per_match
18      Germany                2.248785
17      England                2.184743
10       Brazil                2.173132
38  Netherlands                2.096921
15        Spain                2.046095

Strongest defense (goals conceded per match, min. 300 matches):
           team  goals_co

## Largest winning margins and highest-scoring matches

In [9]:
print("Largest winning margins ever recorded:")
print(matches.sort_values("absolute_goal_difference", ascending=False).head(5)[
    ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]])
print()
print("Highest-scoring matches (total goals):")
print(matches.sort_values("total_goals", ascending=False).head(5)[
    ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]])


Largest winning margins ever recorded:
            date  home_team       away_team  home_score  away_score  \
25420 2001-04-11  Australia  American Samoa          31           0   
8547  1971-09-13     Tahiti    Cook Islands          30           0   
11913 1979-08-30       Fiji        Kiribati          24           0   
25417 2001-04-09  Australia           Tonga          22           0   
6576  1966-04-03      Libya            Oman          21           0   

                         tournament  
25420  FIFA World Cup qualification  
8547            South Pacific Games  
11913           South Pacific Games  
25417  FIFA World Cup qualification  
6576                       Arab Cup  

Highest-scoring matches (total goals):
            date  home_team       away_team  home_score  away_score  \
25420 2001-04-11  Australia  American Samoa          31           0   
8547  1971-09-13     Tahiti    Cook Islands          30           0   
11913 1979-08-30       Fiji        Kiribati          

## Neutral vs. non-neutral venues, and tournament frequency

In [10]:
home_win_neutral = matches.groupby("neutral_match")["home_win"].mean()
draw_rate_neutral = matches.groupby("neutral_match")["is_draw"].mean()
print("Home-team win rate, by neutral-venue flag:")
print(home_win_neutral)
print()
print("Draw rate, by neutral-venue flag:")
print(draw_rate_neutral)

ct = pd.crosstab(matches["neutral_match"], matches["home_win"])
chi2, p, dof, exp = stats.chi2_contingency(ct)
print()
print(f"Chi-square test of independence (neutral venue vs. home win): chi2={chi2:.2f}, p={p:.2e}")


Home-team win rate, by neutral-venue flag:
neutral_match
False    0.507161
True     0.441353
Name: home_win, dtype: float64

Draw rate, by neutral-venue flag:
neutral_match
False    0.228621
True     0.224414
Name: is_draw, dtype: float64

Chi-square test of independence (neutral venue vs. home win): chi2=165.32, p=7.80e-38


**Statistical analysis — Home-field advantage**

*Research question:* Is the "home team" win rate different on neutral grounds vs.
true home venues?
*Hypothesis:* H0: home-win rate is independent of whether the venue is neutral.
*Result:* Home teams win 50.7% of matches at their own/non-neutral venues vs. 44.1% on
neutral grounds, a difference confirmed by a chi-square test (p << 0.001). This is
consistent with a real home-field advantage (crowd, travel, familiarity) rather than
random noise, though this analysis is descriptive/associational -- it does not control
for opponent strength, era, or competition type, so we do not claim a precise causal
effect size.

In [11]:
print("Most frequent tournament types:")
print(matches["tournament_category"].value_counts())
print()
print("Top 10 individual tournament names by match count:")
print(matches["tournament"].value_counts().head(10))


Most frequent tournament types:
tournament_category
Friendly                        18252
Other                           14943
World Cup qualification          8771
Continental competition          6285
World Cup (final tournament)      964
Name: count, dtype: int64

Top 10 individual tournament names by match count:
tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64


## Correlation analysis: do these metrics actually relate the way we assume?

In [12]:
r1, p1 = stats.pearsonr(main_pool["win_rate"], main_pool["avg_goal_diff"])
print(f"win_rate vs avg_goal_diff: r={r1:.3f}, p={p1:.2e}, n={len(main_pool)}")

r2, p2 = stats.pearsonr(main_pool["dominant_win_rate"], main_pool["points_per_match"])
print(f"dominant_win_rate vs points_per_match: r={r2:.3f}, p={p2:.2e}, n={len(main_pool)}")


win_rate vs avg_goal_diff: r=0.972, p=9.47e-85, n=133
dominant_win_rate vs points_per_match: r=0.859, p=7.84e-40, n=133


*Interpretation:* Win rate and average goal differential are almost perfectly
correlated (r=0.97) among teams with at least 300 matches -- unsurprising, since both
are downstream of the same underlying scoring dominance, but it confirms the two metrics
are not contributing meaningfully independent information when used together (a
limitation noted explicitly in the Team Dominance Index discussion in notebook 04).
Dominant-win rate and points-per-match are also strongly related (r=0.86) but less
redundant, since a team can accumulate points through narrow wins/draws without ever
recording a 3+ goal margin.

This completes the all-international-matches exploratory analysis. Notebook 03 repeats
and extends a parallel set of analyses restricted to FIFA World Cup final tournaments
from 2002 onward, plus the goalscorer supplementary analysis.